In [118]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns',None)

In [119]:
df1 = pd.read_csv("../data/diet_recommendations_dataset.csv")
df1.head()



,Patient_ID,Age,Gender,Weight_kg,Height_cm,BMI,Disease_Type,Severity,Physical_Activity_Level,Daily_Caloric_Intake,Cholesterol_mg/dL,Blood_Pressure_mmHg,Glucose_mg/dL,Dietary_Restrictions,Allergies,Preferred_Cuisine,Weekly_Exercise_Hours,Adherence_to_Diet_Plan,Dietary_Nutrient_Imbalance_Score,Diet_Recommendation
0,P0001,56,Male,58.4,160,22.8,Obesity,Moderate,Moderate,3079,173.3,133,116.3,NaN,Peanuts,Mexican,3.1,96.6,3.1,Balanced
1,P0002,69,Male,101.2,169,35.4,Diabetes,Mild,Moderate,3032,199.2,120,137.1,NaN,Peanuts,Chinese,4.5,63.2,0.6,Low_Carb
2,P0003,46,Female,63.5,173,21.2,Hypertension,Mild,Sedentary,1737,181.0,121,109.6,NaN,Peanuts,Chinese,3.8,57.5,4.6,Low_Sodium
3,P0004,32,Male,58.1,164,21.6,NaN,Mild,Moderate,2657,168.2,144,159.4,NaN,NaN,Mexican,4.3,54.5,0.4,Balanced
4,P0005,60,Male,79.5,197,20.5,Diabetes,Moderate,Sedentary,3496,200.4,172,182.3,Low_Sugar,NaN,Italian,9.8,78.2,4.7,Low_Carb


In [120]:
import pandas as pd

import pandas as pd

# Put ALL files in the same folder and use consistent path
path = '../data/USER HEALTH/'


# ── STEP 1: Load all files ──
demo  = pd.read_sas(path + 'DEMO_J.xpt')
bmx   = pd.read_sas(path + 'BMX_J.xpt')
bpx   = pd.read_sas(path + 'BPX_J.xpt')
tchol = pd.read_sas(path + 'TCHOL_J.xpt')
paq   = pd.read_sas(path + 'PAQ_J.xpt')
dr1   = pd.read_sas(path + 'DR1TOT_J.xpt')
mcq   = pd.read_sas(path + 'MCQ_J.xpt')
diq   = pd.read_sas(path + 'DIQ_J.xpt')
 # ← was missing

# ── STEP 2: Extract important columns ──
demo  = demo[['SEQN', 'RIDAGEYR', 'RIAGENDR']]
bmx   = bmx[['SEQN',  'BMXWT', 'BMXHT', 'BMXBMI']]
bpx   = bpx[['SEQN',  'BPXSY1', 'BPXDI1']]
tchol = tchol[['SEQN', 'LBXTC']]
paq   = paq[['SEQN',  'PAQ605']]
dr1   = dr1[['SEQN',  'DR1TKCAL', 'DR1TPROT', 'DR1TCARB', 'DR1TTFAT', 'DR1TSUGR', 'DR1TSODI', 'DR1TFIBE']]
mcq   = mcq[['SEQN',  'MCQ160B', 'MCQ160L', 'MCQ220']]
diq   = diq[['SEQN',  'DIQ010']]


# ── STEP 3: Merge all ──
df = demo.merge(bmx,   on='SEQN', how='left') \
         .merge(bpx,   on='SEQN', how='left') \
         .merge(tchol, on='SEQN', how='left') \
         .merge(paq,   on='SEQN', how='left') \
         .merge(dr1,   on='SEQN', how='left') \
         .merge(mcq,   on='SEQN', how='left') \
         .merge(diq,   on='SEQN', how='left') \
        

print(df.shape)
print(df.columns.tolist())

(9254, 21)
['SEQN', 'RIDAGEYR', 'RIAGENDR', 'BMXWT', 'BMXHT', 'BMXBMI', 'BPXSY1', 'BPXDI1', 'LBXTC', 'PAQ605', 'DR1TKCAL', 'DR1TPROT', 'DR1TCARB', 'DR1TTFAT', 'DR1TSUGR', 'DR1TSODI', 'DR1TFIBE', 'MCQ160B', 'MCQ160L', 'MCQ220', 'DIQ010']


In [121]:
# See what columns exist before rename
print([c for c in ['DIQ010', 'BPQ020'] if c in df.columns])

['DIQ010']


In [122]:
df.rename(columns={
    'SEQN'    : 'Patient_ID',
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'BMXWT'   : 'Weight_kg',
    'BMXHT'   : 'Height_cm',
    'BMXBMI'  : 'BMI',
    'BPXSY1'  : 'BP_Systolic',
    'BPXDI1'  : 'BP_Diastolic',
    'LBXTC'   : 'Cholesterol_mg_dL',
    'PAQ605'  : 'Physical_Activity',        # ← LBXGLU line removed
    'DR1TKCAL': 'Daily_Calories',
    'DR1TPROT': 'Daily_Protein_g',
    'DR1TCARB': 'Daily_Carbs_g',
    'DR1TTFAT': 'Daily_Fat_g',
    'DR1TSUGR': 'Daily_Sugar_g',
    'DR1TSODI': 'Daily_Sodium_mg',
    'DR1TFIBE': 'Daily_Fiber_g',
    'DIQ010'  : 'Has_Diabetes',
    'BPQ020'  : 'Has_Hypertension',
    'MCQ160B' : 'Has_HeartDisease',
    'MCQ160L' : 'Has_Liver_Condition',
    'MCQ220'  : 'Has_Cancer',
}, inplace=True)


# Instead derive Has_Hypertension from actual BP readings
df['Has_Hypertension'] = ((df['BP_Systolic'] >= 130) | (df['BP_Diastolic'] >= 80)).astype(int)

df['Gender'] = df['Gender'].map({1: 'Male', 2: 'Female'})
df['Physical_Activity'] = df['Physical_Activity'].map({1: 'Active', 2: 'Sedentary'})

for col in ['Has_Diabetes', 'Has_Hypertension', 'Has_HeartDisease', 'Has_Liver_Condition', 'Has_Cancer']:
    df[col] = df[col].map({1: 1, 2: 0, 3: 0})

df['Blood_Pressure_mmHg'] = df['BP_Systolic'].astype(str) + '/' + df['BP_Diastolic'].astype(str)
df.drop(columns=['BP_Systolic', 'BP_Diastolic'], inplace=True)

def get_disease(row):
    if row['Has_Diabetes'] == 1:         return 'Diabetes'
    if row['Has_Hypertension'] == 1:     return 'Hypertension'
    if row['Has_HeartDisease'] == 1:     return 'Heart Disease'
    if row['Has_Liver_Condition'] == 1:  return 'Liver Condition'
    if row['Has_Cancer'] == 1:           return 'Cancer'
    if row['BMI'] >= 30:                 return 'Obesity'
    return 'None'

df['Disease_Type'] = df.apply(get_disease, axis=1)

df.dropna(subset=['BMI'], inplace=True)

# print(df.shape)
# print(df['Disease_Type'].value_counts())
df = df[df['Age'] >= 18]
print(df.columns.tolist())
# df.to_csv(path + 'nhanes_clean.csv', index=False)

['Patient_ID', 'Age', 'Gender', 'Weight_kg', 'Height_cm', 'BMI', 'Cholesterol_mg_dL', 'Physical_Activity', 'Daily_Calories', 'Daily_Protein_g', 'Daily_Carbs_g', 'Daily_Fat_g', 'Daily_Sugar_g', 'Daily_Sodium_mg', 'Daily_Fiber_g', 'Has_HeartDisease', 'Has_Liver_Condition', 'Has_Cancer', 'Has_Diabetes', 'Has_Hypertension', 'Blood_Pressure_mmHg', 'Disease_Type']


In [123]:
round(df.isnull().sum()/len(df)*100,2).sort_values(ascending=False)

Has_Hypertension       59.37
Blood_Pressure_mmHg    11.06
Daily_Fat_g             9.39
Daily_Calories          9.39
Daily_Protein_g         9.39
Daily_Carbs_g           9.39
Daily_Sugar_g           9.39
Daily_Sodium_mg         9.39
Daily_Fiber_g           9.39
Cholesterol_mg_dL       6.29
Has_Liver_Condition     5.02
Has_HeartDisease        5.01
Has_Cancer              4.80
Physical_Activity       0.11
Has_Diabetes            0.07
Patient_ID              0.00
Age                     0.00
BMI                     0.00
Height_cm               0.00
Weight_kg               0.00
Gender                  0.00
Disease_Type            0.00
dtype: float64

In [124]:
def get_diet_recommendation(row):
    if row['Has_Diabetes'] == 1:     return 'Diabetic'
    if row['Has_HeartDisease'] == 1: return 'Heart Healthy'
    if row['Has_Hypertension'] == 1: return 'Low Sodiumt'
    if row['BMI'] >= 35:             return 'Very Low'
    if row['BMI'] >= 30:             return 'Low Calorie'
    if row['BMI'] < 18.5:            return 'High Calorie'
    return 'Balanced'

df['Diet_Recommendation'] = df.apply(get_diet_recommendation, axis=1)
df.head(3)

,Patient_ID,Age,Gender,Weight_kg,Height_cm,BMI,Cholesterol_mg_dL,Physical_Activity,Daily_Calories,Daily_Protein_g,Daily_Carbs_g,Daily_Fat_g,Daily_Sugar_g,Daily_Sodium_mg,Daily_Fiber_g,Has_HeartDisease,Has_Liver_Condition,Has_Cancer,Has_Diabetes,Has_Hypertension,Blood_Pressure_mmHg,Disease_Type,Diet_Recommendation
2,93705.0,66.0,Female,79.5,158.3,31.7,157.0,Sedentary,1202.0,20.01,157.45,56.98,91.55,3574.0,8.4,0.0,0.0,0.0,0.0,NaN,NaN,Obesity,Low Calorie
3,93706.0,18.0,Male,66.3,175.7,21.5,148.0,Sedentary,1987.0,94.19,89.82,137.39,14.73,3657.0,7.1,NaN,NaN,NaN,0.0,NaN,112.0/74.0,None,Balanced
5,93708.0,66.0,Female,53.5,150.2,23.7,209.0,Sedentary,1251.0,50.96,123.71,65.49,49.84,2135.0,16.6,0.0,0.0,0.0,0.0,NaN,NaN,None,Balanced
